In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BronzeToSilver") \
    .config(
        "spark.hadoop.hive.metastore.uris",
        "thrift://docker-hive-hive-metastore-1:9083"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

In [9]:
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|  default|
|warehouse|
+---------+



In [10]:
bronze_df = spark.read.parquet(
    "hdfs://docker-hive-namenode-1:8020/data-lake/bronze/transaksi"
)

bronze_df.show()

+---+----------+------+----------+--------+------+-------+
| id|   tanggal|produk|  kategori|    kota|jumlah|  harga|
+---+----------+------+----------+--------+------+-------+
|  1|2026-05-01|Laptop|Elektronik| Jakarta|     1|8000000|
|  2|2026-05-01| Mouse|Elektronik| Bandung|     2| 150000|
|  3|2026-05-02|  Buku|   Edukasi|Surabaya|     3|  75000|
|  4|2026-05-02|Laptop|Elektronik| Jakarta|     1|8000000|
|  5|2026-05-03|Pulpen|   Edukasi|  Kediri|    10|   5000|
|  6|2026-05-03|Laptop|Elektronik| Jakarta|     1|8000000|
|  7|2026-05-04| Mouse|Elektronik| Bandung|     1| 150000|
|  8|2026-05-04|  Buku|   Edukasi|Surabaya|     2|  75000|
+---+----------+------+----------+--------+------+-------+



In [11]:
silver_df = bronze_df \
    .dropDuplicates(["id"]) \
    .withColumn("produk", trim(col("produk"))) \
    .withColumn("kategori", trim(col("kategori"))) \
    .withColumn("kota", trim(col("kota"))) \
    .filter(col("id").isNotNull()) \
    .filter(col("tanggal").isNotNull()) \
    .filter(col("produk").isNotNull()) \
    .filter(col("jumlah") > 0) \
    .filter(col("harga") > 0)

silver_df.show()
silver_df.printSchema()

+---+----------+------+----------+--------+------+-------+
| id|   tanggal|produk|  kategori|    kota|jumlah|  harga|
+---+----------+------+----------+--------+------+-------+
|  1|2026-05-01|Laptop|Elektronik| Jakarta|     1|8000000|
|  2|2026-05-01| Mouse|Elektronik| Bandung|     2| 150000|
|  3|2026-05-02|  Buku|   Edukasi|Surabaya|     3|  75000|
|  4|2026-05-02|Laptop|Elektronik| Jakarta|     1|8000000|
|  5|2026-05-03|Pulpen|   Edukasi|  Kediri|    10|   5000|
|  6|2026-05-03|Laptop|Elektronik| Jakarta|     1|8000000|
|  7|2026-05-04| Mouse|Elektronik| Bandung|     1| 150000|
|  8|2026-05-04|  Buku|   Edukasi|Surabaya|     2|  75000|
+---+----------+------+----------+--------+------+-------+

root
 |-- id: integer (nullable = true)
 |-- tanggal: date (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- jumlah: integer (nullable = true)
 |-- harga: integer (nullable = true)



In [12]:
silver_df = silver_df.withColumn(
    "revenue",
    col("jumlah") * col("harga")
)

silver_df.show()

+---+----------+------+----------+--------+------+-------+-------+
| id|   tanggal|produk|  kategori|    kota|jumlah|  harga|revenue|
+---+----------+------+----------+--------+------+-------+-------+
|  1|2026-05-01|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  2|2026-05-01| Mouse|Elektronik| Bandung|     2| 150000| 300000|
|  3|2026-05-02|  Buku|   Edukasi|Surabaya|     3|  75000| 225000|
|  4|2026-05-02|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  5|2026-05-03|Pulpen|   Edukasi|  Kediri|    10|   5000|  50000|
|  6|2026-05-03|Laptop|Elektronik| Jakarta|     1|8000000|8000000|
|  7|2026-05-04| Mouse|Elektronik| Bandung|     1| 150000| 150000|
|  8|2026-05-04|  Buku|   Edukasi|Surabaya|     2|  75000| 150000|
+---+----------+------+----------+--------+------+-------+-------+



In [13]:
silver_path = "hdfs://docker-hive-namenode-1:8020/data-lake/silver/transaksi"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)